In [3]:
# =============================================
# 1. 더미 뉴스 데이터 생성
# =============================================

dummy_news = [
    {"text": "삼성전자, 반도체 투자 확대 발표", "stocks": ["삼성전자"]},
    {"text": "SK하이닉스, HBM 수요 증가 소식", "stocks": ["SK하이닉스"]},
    {"text": "LG화학, 배터리 신기술 개발", "stocks": ["LG화학"]},
    {"text": "삼성전자와 SK하이닉스, 공급망 협력 강화", "stocks": ["삼성전자", "SK하이닉스"]},
    {"text": "LG화학과 삼성전자 소재 협약 체결", "stocks": ["LG화학", "삼성전자"]},
]

tickers = ["삼성전자", "SK하이닉스", "LG화학"]


In [4]:
# =============================================
# 2. 더미 임베딩 함수
# =============================================
import torch
import numpy as np

def dummy_embedding(text):
    # 텍스트 길이를 반영해도 되고, 지금은 임시로 랜덤 8차원 벡터
    return torch.tensor(np.random.rand(8), dtype=torch.float32)


In [5]:
# =============================================
# 3. 종목별 node feature 생성 (뉴스 임베딩 평균)
# =============================================

node_features = {t: [] for t in tickers}

for news in dummy_news:
    emb = dummy_embedding(news["text"])
    for stock in news["stocks"]:
        node_features[stock].append(emb)

# 평균 pooling
for stock in node_features:
    node_features[stock] = torch.stack(node_features[stock]).mean(dim=0)

node_features


{'삼성전자': tensor([0.6006, 0.3713, 0.6876, 0.8920, 0.4509, 0.7595, 0.6594, 0.5551]),
 'SK하이닉스': tensor([0.5782, 0.3570, 0.5683, 0.9784, 0.7836, 0.6898, 0.5810, 0.3987]),
 'LG화학': tensor([0.3796, 0.7951, 0.3577, 0.3779, 0.5532, 0.6386, 0.7672, 0.8337])}

In [6]:
# =============================================
# 4. edge_index 생성 (공동 언급 뉴스 기반)
# =============================================

edges = []

for news in dummy_news:
    stocks = news["stocks"]
    if len(stocks) > 1:
        for i in range(len(stocks)):
            for j in range(i+1, len(stocks)):
                # undirected edge
                edges.append((stocks[i], stocks[j]))
                edges.append((stocks[j], stocks[i]))

stock_to_id = {name: idx for idx, name in enumerate(tickers)}

edge_index = torch.tensor([
    [stock_to_id[s] for s, t in edges],
    [stock_to_id[t] for s, t in edges]
], dtype=torch.long)

edge_index


tensor([[0, 1, 2, 0],
        [1, 0, 0, 2]])

In [7]:
# =============================================
# 5. GCN 모델 정의 및 실행
# =============================================
from torch_geometric.nn import GCNConv

class MiniGCN(torch.nn.Module):
    def __init__(self, in_dim=8, hid_dim=16, out_dim=8):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hid_dim)
        self.conv2 = GCNConv(hid_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = MiniGCN()

X = torch.stack([node_features[t] for t in tickers])  # shape [3, 8]

output = model(X, edge_index)
output




tensor([[-0.2769,  0.6656, -1.0392,  1.0475,  0.3356, -0.6880,  0.5621,  0.0458],
        [-0.2649,  0.6164, -0.9065,  0.8587,  0.3492, -0.5729,  0.5248,  0.0407],
        [-0.1983,  0.4759, -0.7976,  0.8503,  0.2115, -0.5311,  0.4024,  0.0311]],
       grad_fn=<AddBackward0>)

In [8]:
# =============================================
# 6. 종목 간 임베딩 유사도 확인 (관계 반영 여부 테스트)
# =============================================
import torch.nn.functional as F

def sim(a, b):
    return F.cosine_similarity(a, b, dim=0).item()

print("삼성전자 vs SK하이닉스:", sim(output[0], output[1]))
print("삼성전자 vs LG화학:", sim(output[0], output[2]))
print("SK하이닉스 vs LG화학:", sim(output[1], output[2]))


삼성전자 vs SK하이닉스: 0.9981785416603088
삼성전자 vs LG화학: 0.9984533190727234
SK하이닉스 vs LG화학: 0.9933764934539795
